# One row per gene: joining every per-gene result, and which method yields the most biology (notebook 112)

**The question.** Twenty-odd notebooks have each written their own per-gene table and none has been joined to
another. Two things follow and neither has been measured: which genes are carried by several *independent* lines
of evidence beyond chance, and which screen delivers the most clean, externally supported, interpretable genes.

**The frozen protocol** is `docs/results/pt-gene-atlas.md` (commit 14e685d, amendment d4c863f).
- Q1 convergence against an expression-stratified null, with a relabeled control.
- Q2 how independent the lines are.
- Q3 the yield profile of each calling method.
- Q4 the core list and the novel tail.
- Q5 recovery of the source notebooks' own counts, read first.

This notebook fits no model on expression. Every statistic it uses was computed by an earlier notebook.

**Sections.** 1 setup; 2 the AKI atlas; 3 the species atlas; 4 Q5 recovery; 5 Q1 convergence; 6 Q2 similarity;
7 Q3 yield; 8 Q4 core list; 9 figures; 10 reading and run record.

## 1 · Setup and inputs

In [ ]:
import argparse
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
output = results_root / 'pt_gene_atlas'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '112.gene_atlas.1'
SEED = 112
DRAWS, KMAX, N_DECILES = 2000, 6, 10
EXTERNAL_T, Q_CUT, L_CUT, EDGE_Q = 2.0, .05, 1.0, .95
L91_CUT, M91_CUT = .5, .4                      # notebook 91's own level and location thresholds

SRC = {
    '91': results_root / 'pt_positional_decomposition' / 'tables',
    '94': results_root / 'pt_external_iri' / 'tables',
    '97': results_root / 'pt_coexpression_conservation' / 'tables',
    '98': results_root / 'pt_human_aki_benchmark' / 'tables',
    '99': results_root / 'pt_injury_phases' / 'tables',
    '101': results_root / 'pt_transporter_handoffs' / 'tables',
    '104': results_root / 'pt_contamination_audit' / 'tables',
    '105': results_root / 'pt_external_spatial_iri' / 'tables',
    '106': results_root / 'pt_gradient_evolution' / 'tables',
    '107': results_root / 'pt_injury_heterogeneity' / 'tables',
    '108': results_root / 'pt_disease_genetics' / 'tables',
    '109': results_root / 'pt_neighbour_marker_filter' / 'tables',
    '111': results_root / 'pt_androgen_s3' / 'tables',
    'ftest': results_root / 'pt_normalized_t_spatial' / 'tables',
    'rown': results_root / 'pt_pathway_networks' / 'tables',
    'species': results_root / 'pt_species_level_control' / 'tables',
    'zon': results_root / 'pt_zonation_classes' / 'tables',
    'known_aki': results_root / 'pt_known_injury_programs' / 'tables',
    'known_spc': results_root / 'pt_known_species_programs' / 'tables',
    'univ': results_root / 'pt_pathways_package_pseudospace' / 'tables',
}
NEEDED = [
    SRC['91'] / 'i_genes.csv.gz', SRC['94'] / 'external_genes.csv.gz', SRC['97'] / 'genes.csv',
    SRC['97'] / 'post_hoc_gene_lists.csv', SRC['98'] / 'external_genes.csv.gz', SRC['99'] / 'phase_programs.csv',
    SRC['99'] / 'gene_effects.csv.gz', SRC['101'] / 'pairs.csv', SRC['101'] / 'eligible.csv',
    SRC['104'] / 'aki_robustness.csv.gz', SRC['104'] / 'gradient_robustness.csv.gz',
    SRC['104'] / 'edge_enrichment.csv.gz', SRC['105'] / 'positional_genes.csv', SRC['106'] / 'genes.csv',
    SRC['107'] / 'genes.csv.gz', SRC['108'] / 'sets.csv', SRC['108'] / 'h4_named.csv',
    SRC['109'] / 'posthoc_suspect.csv', SRC['111'] / 'sets.csv',
    SRC['ftest'] / 'gene_change_aki_vs_control.csv', SRC['ftest'] / 'gene_change_human_vs_mouse.csv',
    SRC['rown'] / 'row_n_genes_aki_vs_control.csv', SRC['species'] / 'gene_level.csv',
    SRC['zon'] / 'gene_zonation_classes.csv', SRC['known_aki'] / 'programs_members.csv',
    SRC['known_spc'] / 'programs_members.csv', SRC['univ'] / 'universe_human_vs_mouse.csv',
]
for path in NEEDED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from IPython.display import display

from pseudospace import gene_atlas as ga

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
BLUE, ORANGE, GREEN, GREY = '#0072B2', '#D55E00', '#009E73', '#999999'
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


def deciles(values, n=N_DECILES):
    """Expression strata: equal-count bins of a value, missing values in their own stratum."""
    v = pd.Series(values).astype(float)
    out = pd.Series(-1, index=v.index, dtype=int)
    ok = v.notna()
    out[ok] = pd.qcut(v[ok].rank(method='first'), n, labels=False).astype(int)
    return out


def beyond_others(stat, others, cutoff, ratio=.5):
    """Notebook 91's calibrated rule with ``stat`` as the contrast: |stat| >= cutoff and each of ``others`` at most
    ``ratio`` of it. Applied with a relabeling in the contrast's place, it is that notebook's own I1 control."""
    s = pd.Series(stat).astype(float).abs()
    ok = s >= cutoff
    for o in others:
        ok &= pd.Series(o).astype(float).abs() <= ratio * s
    return ok.fillna(False)


def flag_above_matched(values, strata, q=EDGE_Q):
    """True where a value exceeds the q-th percentile of its own stratum (missing values are never flagged)."""
    v, s = pd.Series(values).astype(float), pd.Series(strata)
    out = pd.Series(False, index=v.index)
    for g, idx in v.groupby(s).groups.items():
        sub = v.loc[idx].dropna()
        if len(sub) >= 20:
            out.loc[sub.index] = sub > sub.quantile(q)
    return out


guard_log = []

## 2 · The AKI atlas

Universe: notebook 91's shared AKI universe. Expression strata: its control log2 CP10k.

In [ ]:
g91 = pd.read_csv(SRC['91'] / 'i_genes.csv.gz').set_index('gene')
g91 = g91[~g91.index.duplicated()]
AKI = g91.index.astype(str)
aki_expr = g91['log2 CP10k (controls)']
aki_strata = deciles(aki_expr)
guard_log.append(f'G1: AKI universe {len(AKI):,} genes from notebook 91')

ftest_aki = pd.read_csv(SRC['ftest'] / 'gene_change_aki_vs_control.csv').set_index('gene')
rown_aki = pd.read_csv(SRC['rown'] / 'row_n_genes_aki_vs_control.csv').set_index('gene')
kir = pd.read_csv(SRC['94'] / 'external_genes.csv.gz', header=[0, 1], index_col=0)
kpmp = pd.read_csv(SRC['98'] / 'external_genes.csv.gz', header=[0, 1], index_col=0)
phase = pd.read_csv(SRC['99'] / 'phase_programs.csv')
phase_universe = pd.read_csv(SRC['99'] / 'gene_effects.csv.gz', usecols=['gene'])['gene']
het = pd.read_csv(SRC['107'] / 'genes.csv.gz').set_index('gene')
ar = pd.read_csv(SRC['111'] / 'sets.csv', index_col=0)
ar.index.name = 'gene'

sign_L = np.sign(g91['L'])
kir_t = kir[('12hours', 'level t')].reindex(AKI)
kir_c = kir[('12hours', 'level coefficient')].reindex(AKI)
kpmp_t = kpmp[('primary', 'level t')].reindex(AKI)
kpmp_c = kpmp[('primary', 'level coefficient')].reindex(AKI)
ar_maintained = ar[['maintained S1', 'maintained S2', 'maintained S3']].any(axis=1)

AKI_LINES = {
    'A1 level': (g91.index[g91['level change'].astype(bool)], AKI),
    'A2 location': (g91.index[g91['location change'].astype(bool)], AKI),
    'A3 F-test': (ftest_aki.index[(ftest_aki['q'] < Q_CUT) & ftest_aki['converged'].astype(bool)], ftest_aki.index),
    'A4 position-split': (rown_aki.index[rown_aki['selected (row N)'].astype(bool)],
                          rown_aki.index[rown_aki['usable'].astype(bool)]),
    'A5 Kirita 12 h': (AKI[(kir_t.abs() >= EXTERNAL_T) & (np.sign(kir_c) == sign_L) & sign_L.ne(0)],
                       AKI[kir_t.notna()]),
    'A6 KPMP': (AKI[(kpmp_t.abs() >= EXTERNAL_T) & (np.sign(kpmp_c) == sign_L) & sign_L.ne(0)],
                AKI[kpmp_t.notna()]),
    'A7 phase': (phase.loc[phase['program'].str.startswith(('ACUTE', 'SUSTAINED')), 'gene'], phase_universe),
    'A8 variability': (het.index[het['beyond both relabelings'].astype(bool)], het.index),
    'A9 androgen': (ar.index[ar_maintained & (ar['L'] < 0)], ar.index),
}
aki_calls, aki_testable = ga.join_lines(AKI, AKI_LINES)
print(pd.DataFrame({'called': aki_calls.sum(), 'testable': aki_testable.sum()}))

**Filters.** F1 core rebuild, F2 neighbour marker, F3 edge enrichment. A gene that notebook 104 never tested
cannot fail F1, so "survives" means "not failing".

In [ ]:
rob_aki = pd.read_csv(SRC['104'] / 'aki_robustness.csv.gz')
fails_core = set(rob_aki.loc[~rob_aki['survives core3'].astype(bool), 'gene'].astype(str))
suspect_rows = pd.read_csv(SRC['109'] / 'posthoc_suspect.csv')
SUSPECT = set()
for s in suspect_rows['suspect genes'].dropna():
    SUSPECT |= {t.strip() for t in str(s).split(',') if t.strip()}
edge = pd.read_csv(SRC['104'] / 'edge_enrichment.csv.gz').set_index('gene')
edge = edge[~edge.index.duplicated()]
aki_edge = edge[[c for c in edge.columns if c.startswith('aki ')]].mean(axis=1).reindex(AKI)
aki_edge_flag = flag_above_matched(aki_edge, aki_strata)

aki_clean = pd.Series(True, index=AKI)
aki_clean &= ~AKI.isin(fails_core)
aki_clean &= ~AKI.isin(SUSPECT) & ~AKI.str.upper().isin({s.upper() for s in SUSPECT})
aki_clean &= ~aki_edge_flag.reindex(AKI).fillna(False).to_numpy()
aki_supported = (aki_calls['A5 Kirita 12 h'] | aki_calls['A6 KPMP'])
print(f'clean {aki_clean.sum():,} of {len(AKI):,} · supported {aki_supported.sum():,}')

**Annotation.** Named membership for the "anchored" criterion.

In [ ]:
known_aki = pd.read_csv(SRC['known_aki'] / 'programs_members.csv')
transporters = pd.read_csv(SRC['101'] / 'eligible.csv', index_col=0)
disease = pd.read_csv(SRC['108'] / 'sets.csv')
named = pd.read_csv(SRC['108'] / 'h4_named.csv')
pos105 = pd.read_csv(SRC['105'] / 'positional_genes.csv')
ANN_AKI = {
    'literature program': set(known_aki['gene'].astype(str)),
    'transporter': set(transporters.index.astype(str)),
    'disease gene': set(disease['mouse'].dropna().astype(str)),
    'androgen set': set(ar.index[ar[[c for c in ar.columns if c.startswith(('maintained', 'repressed'))]].any(axis=1)]),
    'positional marker': set(pos105['gene'].astype(str)),
    'named position': {str(x).capitalize() for x in named['gene'].astype(str)},
}
aki_ann = pd.DataFrame({k: AKI.isin(v) for k, v in ANN_AKI.items()}, index=AKI)
aki_anchored = aki_ann.any(axis=1)
print(aki_ann.sum().to_dict(), '· anchored', int(aki_anchored.sum()))

## 3 · The species atlas

In [ ]:
univ = pd.read_csv(SRC['univ'] / 'universe_human_vs_mouse.csv').set_index('gene')
univ = univ[univ['eligible'].astype(bool)]
SPC = univ.index.astype(str)
spc_strata = deciles(np.log1p(univ['mean_expression']))
guard_log.append(f'G2: species universe {len(SPC):,} eligible genes')

sl = pd.read_csv(SRC['species'] / 'gene_level.csv').set_index('gene')
sl = sl[~sl.index.duplicated()]
ftest_spc = pd.read_csv(SRC['ftest'] / 'gene_change_human_vs_mouse.csv').set_index('gene')
zon = pd.read_csv(SRC['zon'] / 'gene_zonation_classes.csv').set_index('gene')
rew = pd.read_csv(SRC['97'] / 'post_hoc_gene_lists.csv')
g97 = pd.read_csv(SRC['97'] / 'genes.csv').set_index('gene')
g106 = pd.read_csv(SRC['106'] / 'genes.csv').set_index('mouse')
pairs = pd.read_csv(SRC['101'] / 'pairs.csv')

CONTRASTS = ['S2-S1', 'S3c-early', 'S3-early']
disagree = pd.Series(False, index=zon.index)
tested = pd.Series(False, index=zon.index)
for c in CONTRASTS:
    m, h = zon[f'{c} mouse call'], zon[f'{c} human call']
    both = m.isin(['up', 'down', 'flat']) & h.isin(['up', 'down', 'flat'])
    tested |= both
    disagree |= both & m.ne(h)

sl_level = (sl['L'].abs() >= L_CUT) & sl['converged'].astype(bool) & ~sl['separated'].astype(bool)
sl_atlas = sl['atlas_strong'].astype(bool) & (np.sign(sl['A']) == np.sign(sl['L'])) & sl['L'].ne(0)
handoff_genes = set(pairs.loc[pairs['confirmed'].astype(bool), 'early'].astype(str)) | \
                set(pairs.loc[pairs['confirmed'].astype(bool), 'late'].astype(str))

SPC_LINES = {
    'S1 level': (sl.index[sl_level], sl.index),
    'S2 F-test': (ftest_spc.index[(ftest_spc['q'] < Q_CUT) & ftest_spc['converged'].astype(bool)], ftest_spc.index),
    'S3 zonation class': (zon.index[disagree], zon.index[tested]),
    'S4 atlas level': (sl.index[sl_atlas], sl.index[sl['atlas_strong'].notna()]),
    'S5 rewired': (rew.loc[rew['group'].eq('rewired in both'), 'gene'], g97.index),
    'S6 reversed gradient': (g106.index[g106['category'].eq('reversed')], g106.index[g106['category'].notna()]),
    'S7 handoff': (pd.Index(sorted(handoff_genes)), transporters.index),
}
spc_calls, spc_testable = ga.join_lines(SPC, SPC_LINES)
print(pd.DataFrame({'called': spc_calls.sum(), 'testable': spc_testable.sum()}))

rob_spc = pd.read_csv(SRC['104'] / 'gradient_robustness.csv.gz')
fails_grad = set(rob_spc.loc[~rob_spc['survives core3'].astype(bool), 'gene'].astype(str))
spc_edge = edge[[c for c in edge.columns if c.startswith('hvm ')]].mean(axis=1).reindex(SPC)
spc_edge_flag = flag_above_matched(spc_edge, spc_strata)
probe_clean = sl['probe_clean_both'].reindex(SPC).eq(True)

spc_clean = pd.Series(True, index=SPC)
spc_clean &= ~SPC.isin(fails_grad) & ~SPC.str.upper().isin({s.upper() for s in fails_grad})
spc_clean &= ~SPC.isin(SUSPECT) & ~SPC.str.upper().isin({s.upper() for s in SUSPECT})
spc_clean &= ~spc_edge_flag.reindex(SPC).fillna(False).to_numpy()
spc_clean &= probe_clean.to_numpy()
spc_supported = spc_calls['S4 atlas level']
print(f'clean {spc_clean.sum():,} of {len(SPC):,} · supported {spc_supported.sum():,}')

known_spc = pd.read_csv(SRC['known_spc'] / 'programs_members.csv')
ANN_SPC = {
    'literature program': set(known_spc['gene'].astype(str)),
    'transporter': set(transporters.index.astype(str)),
    'disease gene': set(disease['mouse'].dropna().astype(str)),
    'handoff': handoff_genes,
    'named position': {str(x).capitalize() for x in named['gene'].astype(str)},
}
spc_ann = pd.DataFrame({k: SPC.isin(v) for k, v in ANN_SPC.items()}, index=SPC)
spc_anchored = spc_ann.any(axis=1)
print(spc_ann.sum().to_dict(), '· anchored', int(spc_anchored.sum()))

## 4 · Q5 · Does the atlas reproduce its sources?

Read first: a mismatch here is a join error.

In [ ]:
RECOVERY = pd.DataFrame([
    ('91 level calls', int(g91['level change'].sum()), '3,159 genes called at the level'),
    ('91 location calls', int(g91['location change'].sum()), 'location beyond level'),
    ('104 AKI calls failing the core rebuild', len(fails_core & set(AKI)), '1.5% of level and 0.3% of location'),
    ('104 species gradients failing the core rebuild', len(fails_grad & set(SPC)), '2% mouse, 3.5% human'),
    ('109 suspect genes', len(SUSPECT), '0-8% of the lists, mostly human late'),
    ('101 confirmed handoff pairs', int(pairs['confirmed'].sum()), '66 pairs across both species'),
    ('111 AR-maintained S3 genes in the AKI universe', int(ar['maintained S3'].sum()), '186 in the shared universe'),
    ('105 injury-stable positional genes', len(pos105), '24 early and 19 late'),
    ('transporters', len(transporters), '237 eligible'),
], columns=['check', 'value', 'expected from the source document'])
RECOVERY.to_csv(tables / 'recovery_checks.csv', index=False)
display(RECOVERY)

## 5 · Q1 · Does evidence converge beyond chance?

In [ ]:
rows = []
NULLS, OBSERVED = {}, {}
for name, (calls, testable, strata) in {
        'AKI': (aki_calls, aki_testable, aki_strata),
        'species': (spc_calls, spc_testable, spc_strata)}.items():
    obs = ga.convergence_counts(calls, testable, kmax=KMAX)
    null = ga.stratified_line_null(calls, testable, strata, draws=DRAWS, seed=SEED, kmax=KMAX)
    OBSERVED[name], NULLS[name] = obs, null
    for k in range(1, KMAX + 1):
        rows.append({'atlas': name, 'k': k, 'observed': int(obs[k]), 'null mean': null[k].mean(),
                     'null 99%': null[k].quantile(.99), 'enrichment': obs[k] / max(null[k].mean(), 1e-9),
                     'beyond 99%': bool(obs[k] > null[k].quantile(.99))})
CONV = pd.DataFrame(rows)
CONV.to_csv(tables / 'convergence.csv', index=False)
display(CONV.round(2))
Q1_HOLDS = bool(CONV[(CONV.k == 3)]['beyond 99%'].all())
print('Q1 holds (k = 3 beyond the 99th percentile in both atlases):', Q1_HOLDS)

**C1 · the relabeled control.** A1, A2, A8 and A9 are rebuilt with a relabeling in the contrast's place, using
each screen's own calibrated rule (notebook 91's |L| ≥ 0.5 and |M| ≥ 0.4 with the two other statistics at most
half the contrast's). The other lines stay as they are, so the control asks what the condition contrast adds.

In [ ]:
C1, C1_SIZES = {}, {}
for tag, other in (('R1', 'R2'), ('R2', 'R1')):
    lines = dict(AKI_LINES)
    lvl = beyond_others(g91[f'L {tag}'], [g91['L'], g91[f'L {other}']], L91_CUT)
    loc = beyond_others(g91[f'M {tag}'], [g91['M'], g91[f'M {other}']], M91_CUT)
    var = beyond_others(het[tag], [het['Δε condition'], het[other]], 0.)
    lines['A1 level'] = (g91.index[lvl], AKI)
    lines['A2 location'] = (g91.index[loc], AKI)
    lines['A8 variability'] = (het.index[var], het.index)
    lines['A9 androgen'] = (ar.index[ar_maintained & (ar[f'L {tag}'] < 0)], ar.index)
    c, t = ga.join_lines(AKI, lines)
    C1[tag] = ga.convergence_counts(c, t, kmax=KMAX)
    C1_SIZES[tag] = {'A1 level': int(lvl.sum()), 'A2 location': int(loc.sum()), 'A8 variability': int(var.sum()),
                     'A9 androgen': int((ar_maintained & (ar[f'L {tag}'] < 0)).sum())}
C1_TABLE = pd.DataFrame({'observed': OBSERVED['AKI'], 'R1': C1['R1'], 'R2': C1['R2'],
                         'null mean': NULLS['AKI'].mean()}).rename_axis('k')
C1_TABLE.to_csv(tables / 'relabeled_convergence.csv')
display(C1_TABLE.round(1))
print('relabeled line sizes against the condition:',
      {k: {'condition': int(aki_calls[k].sum()), **{t: C1_SIZES[t][k] for t in C1_SIZES}} for k in C1_SIZES['R1']})

## 6 · Q2 · How independent are the lines?

In [ ]:
SIM = {}
for name, (calls, testable) in {'AKI': (aki_calls, aki_testable), 'species': (spc_calls, spc_testable)}.items():
    jac, phi = ga.line_similarity(calls, testable)
    SIM[name] = (jac, phi)
    d = 1 - phi.fillna(0).to_numpy()
    np.fill_diagonal(d, 0)
    link = hierarchy.average(squareform((d + d.T) / 2, checks=False))
    groups = hierarchy.fcluster(link, t=.5, criterion='distance')      # phi >= 0.5 joins two lines
    SIM[name] = (jac, phi, pd.Series(groups, index=calls.columns))
    print(f'{name}: {len(set(groups))} line groups at phi 0.5 of {calls.shape[1]} lines')
    display(phi.round(2))
LINE_SIM = pd.concat({k: v[1] for k, v in SIM.items()}, names=['atlas', 'line'])
LINE_SIM.to_csv(tables / 'line_similarity.csv')

# one line per group (the largest universe), as a sensitivity
SENS = []
for name, (calls, testable, strata) in {
        'AKI': (aki_calls, aki_testable, aki_strata),
        'species': (spc_calls, spc_testable, spc_strata)}.items():
    groups = SIM[name][2]
    keep = [testable[groups[groups.eq(g)].index].sum().idxmax() for g in sorted(set(groups))]
    obs = ga.convergence_counts(calls[keep], testable[keep], kmax=KMAX)
    null = ga.stratified_line_null(calls[keep], testable[keep], strata, draws=500, seed=SEED + 1, kmax=KMAX)
    SENS.append({'atlas': name, 'lines kept': len(keep), 'k3 observed': int(obs[3]),
                 'k3 null 99%': null[3].quantile(.99), 'beyond': bool(obs[3] > null[3].quantile(.99))})
SENS = pd.DataFrame(SENS)
SENS.to_csv(tables / 'convergence_sensitivity.csv', index=False)
display(SENS.round(2))

## 7 · Q3 · Which method yields the most interpretable biology?

In [ ]:
AKI_METHODS = ['A1 level', 'A2 location', 'A3 F-test', 'A4 position-split']
SPC_METHODS = ['S1 level', 'S2 F-test', 'S3 zonation class']
YIELD = pd.concat([
    ga.yield_profile(aki_calls, aki_testable, AKI_METHODS, clean=aki_clean, supported=aki_supported,
                     anchored=aki_anchored).assign(atlas='AKI'),
    ga.yield_profile(spc_calls, spc_testable, SPC_METHODS, clean=spc_clean, supported=spc_supported,
                     anchored=spc_anchored).assign(atlas='species'),
], ignore_index=True)
YIELD.to_csv(tables / 'method_yield.csv', index=False)
display(YIELD.round(3))

SENS_T = []
for t in (1.0, 3.0):
    sup = (kir_t.abs() >= t) & (np.sign(kir_c) == sign_L) & sign_L.ne(0)
    sup |= (kpmp_t.abs() >= t) & (np.sign(kpmp_c) == sign_L) & sign_L.ne(0)
    prof = ga.yield_profile(aki_calls, aki_testable, AKI_METHODS, clean=aki_clean,
                            supported=sup.fillna(False), anchored=aki_anchored)
    SENS_T.append(prof.assign(threshold=t))
SENS_T = pd.concat(SENS_T, ignore_index=True)
SENS_T.to_csv(tables / 'method_yield_sensitivity.csv', index=False)
display(SENS_T[['threshold', 'method', 'yield', 'novel yield']])

## 8 · Q4 · The core list and the novel tail

In [ ]:
aki_atlas = pd.concat([
    aki_calls.add_prefix('call '), aki_testable.add_prefix('testable '), aki_ann.add_prefix('is '),
    pd.DataFrame({'lines': aki_calls.sum(axis=1), 'testable lines': aki_testable.sum(axis=1),
                  'clean': aki_clean, 'supported': aki_supported, 'anchored': aki_anchored,
                  'expression decile': aki_strata, 'L': g91['L'], 'M': g91['M'],
                  'Kirita 12 h t': kir_t, 'KPMP t': kpmp_t, 'edge enrichment': aki_edge,
                  'injury class': g91['injury class'], 'injury location': g91['injury location'],
                  'survival': g91['survival']}, index=AKI)], axis=1)
aki_atlas.to_csv(tables / 'aki_atlas.csv.gz', index_label='gene')

spc_atlas = pd.concat([
    spc_calls.add_prefix('call '), spc_testable.add_prefix('testable '), spc_ann.add_prefix('is '),
    pd.DataFrame({'lines': spc_calls.sum(axis=1), 'testable lines': spc_testable.sum(axis=1),
                  'clean': spc_clean, 'supported': spc_supported, 'anchored': spc_anchored,
                  'expression decile': spc_strata, 'L': sl['L'].reindex(SPC), 'A': sl['A'].reindex(SPC),
                  'E mouse': g97['E mouse'].reindex(SPC), 'E human': g97['E human'].reindex(SPC),
                  'edge enrichment': spc_edge, 'zonation class': zon['class'].reindex(SPC)}, index=SPC)], axis=1)
spc_atlas.to_csv(tables / 'species_atlas.csv.gz', index_label='gene')

CORE = {}
for name, atlas in (('AKI', aki_atlas), ('species', spc_atlas)):
    core = atlas[(atlas['lines'] >= 3) & atlas['clean'] & atlas['supported']].sort_values('lines', ascending=False)
    CORE[name] = core
    print(f'{name}: core list {len(core):,} genes · novel tail {int((~core["anchored"]).sum()):,}')
pd.concat({k: v for k, v in CORE.items()}, names=['atlas', 'gene']).to_csv(tables / 'core_list.csv.gz')
NOVEL = pd.concat({k: v[~v['anchored']] for k, v in CORE.items()}, names=['atlas', 'gene'])
NOVEL.to_csv(tables / 'novel_tail.csv')
for name in CORE:
    sub = NOVEL.loc[name] if name in NOVEL.index.get_level_values(0) else NOVEL.iloc[:0]
    print(f'\n{name} novel tail, top by lines:')
    cols = [c for c in ('lines', 'L', 'Kirita 12 h t', 'KPMP t', 'injury class', 'injury location', 'survival',
                        'A', 'E mouse', 'E human', 'zonation class') if c in sub.columns]
    display(sub.head(30)[cols].round(2))

**Post hoc.** The frozen k ≥ 3 bar sits where the convergence enrichment is only about 1.1×. The enrichment rises
with k, so the same list is also reported at the k where it is clearly beyond chance. This threshold was chosen
after seeing the convergence table and is a post-hoc reading.

In [ ]:
STRICT = {}
for name, atlas in (('AKI', aki_atlas), ('species', spc_atlas)):
    rows = []
    for k in range(2, KMAX + 1):
        sub = atlas[(atlas['lines'] >= k) & atlas['clean'] & atlas['supported']]
        e = CONV[(CONV.atlas == name) & (CONV.k == k)]['enrichment'].iloc[0]
        rows.append({'atlas': name, 'k': k, 'enrichment': e, 'genes': len(sub),
                     'anchored': int(sub['anchored'].sum()), 'novel': int((~sub['anchored']).sum())})
    STRICT[name] = pd.DataFrame(rows)
STRICT = pd.concat(STRICT.values(), ignore_index=True)
STRICT.to_csv(tables / 'posthoc_core_by_k.csv', index=False)
display(STRICT.round(2))

POSTHOC = {}
for name, atlas in (('AKI', aki_atlas), ('species', spc_atlas)):
    sub = atlas[(atlas['lines'] >= 5) & atlas['clean'] & atlas['supported']].sort_values('lines', ascending=False)
    POSTHOC[name] = sub
    print(f'{name}: post-hoc strict core (≥ 5 lines) {len(sub)} genes, {int((~sub["anchored"]).sum())} unanchored')
pd.concat(POSTHOC, names=['atlas', 'gene']).to_csv(tables / 'posthoc_strict_core.csv')
for name, sub in POSTHOC.items():
    cols = [c for c in ('lines', 'L', 'Kirita 12 h t', 'KPMP t', 'injury class', 'injury location', 'survival',
                        'A', 'E mouse', 'E human', 'zonation class', 'anchored') if c in sub.columns]
    print(f'\n{name} strict core:')
    display(sub.head(45)[cols].round(2))

In [ ]:
VERDICTS = pd.DataFrame([
    ('Q1 convergence beyond the stratified null at k = 3', Q1_HOLDS),
    ('Q1 relabeled control collapses (R1 and R2 below observed at k = 3)',
     bool(C1_TABLE.loc[3, 'R1'] < C1_TABLE.loc[3, 'observed'] and C1_TABLE.loc[3, 'R2'] < C1_TABLE.loc[3, 'observed'])),
    ('Q2 lines are not one group', bool(all(len(set(SIM[n][2])) > 1 for n in SIM))),
    ('Q4 AKI novel tail under 40 genes (the frozen prediction)', bool((~CORE['AKI']['anchored']).sum() < 40)),
], columns=['verdict', 'holds'])
VERDICTS.to_csv(tables / 'verdicts.csv', index=False)
display(VERDICTS)

## 9 · Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(120 * MM, 45 * MM))
for ax, name in zip(axes, ('AKI', 'species')):
    obs, null = OBSERVED[name], NULLS[name]
    ks = list(range(1, KMAX + 1))
    ax.fill_between(ks, [null[k].quantile(.01) for k in ks], [null[k].quantile(.99) for k in ks],
                    color=GREY, alpha=.25, lw=0)
    ax.plot(ks, [null[k].mean() for k in ks], color=GREY, lw=.8)
    if name == 'AKI':
        for tag in ('R1', 'R2'):
            ax.plot(ks, [C1[tag][k] for k in ks], color=GREEN, lw=.8, ls='--')
    ax.plot(ks, [obs[k] for k in ks], marker='o', ms=2.5, color=ORANGE if name == 'AKI' else BLUE)
    ax.set_yscale('log'); ax.set_xlabel('lines of evidence (≥ k)'); ax.set_title(name)
axes[0].set_ylabel('genes')
save(fig, '112a_convergence')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(160 * MM, 60 * MM), gridspec_kw={'wspace': .9})
for ax, name in zip(axes, ('AKI', 'species')):
    phi = SIM[name][1]
    im = ax.imshow(phi.to_numpy(float), cmap='RdBu_r', vmin=-.6, vmax=.6)
    ax.set_xticks(range(len(phi)), phi.columns, rotation=90)
    ax.set_yticks(range(len(phi)), phi.index)
    ax.set_title(name)
    fig.colorbar(im, ax=ax, fraction=.046, pad=.04)
save(fig, '112b_line_similarity')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(120 * MM, 48 * MM), gridspec_kw={'width_ratios': [4, 3]})
for ax, (name, methods) in zip(axes, (('AKI', AKI_METHODS), ('species', SPC_METHODS))):
    sub = YIELD[YIELD.atlas.eq(name)].set_index('method').reindex(methods)
    x = np.arange(len(sub))
    bars = (('called', LIGHT), ('clean', GREY), ('yield', BLUE), ('novel yield', ORANGE))
    for i, (col, colour) in enumerate(bars):
        ax.bar(x + (i - 1.5) * .2, sub[col], color=colour, width=.19)
    ax.set_xticks(x, [m.split(' ', 1)[1] for m in sub.index], rotation=30, ha='right')
    ax.set_yscale('log'); ax.set_ylim(30, 1e4); ax.set_title(name)
axes[0].set_ylabel('genes')
save(fig, '112c_method_yield')
plt.show()

In [ ]:
top = CORE['AKI'].head(40)
mat = top[[c for c in top.columns if c.startswith('call A')]].astype(float)
fig, ax = plt.subplots(figsize=(90 * MM, 110 * MM))
ax.imshow(mat.to_numpy(), cmap='Blues', vmin=0, vmax=1.4, aspect='auto')
ax.set_xticks(range(mat.shape[1]), [c.replace('call ', '') for c in mat.columns], rotation=90)
ax.set_yticks(range(len(top)), top.index)
save(fig, '112d_core_list')
plt.show()

## 10 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', 'A1 and A2 are notebook 91\'s boolean columns; A7 is a phase program beginning with ACUTE or SUSTAINED.'),
    ('reading', 'A5 and A6 require |t| >= 2 in the external contrast and the same sign as our level L; genes with L = 0 are not called.'),
    ('amendment', 'S1 is |L| >= 1 with converged and not separated, S4 is atlas_strong with the atlas sign, S5 is notebook 97\'s "rewired in both" set (amendment 1, commit d4c863f).'),
    ('reading', 'S3 calls a gene when the mouse and human calls differ on any of the three zonation contrasts, both being up, down or flat.'),
    ('reading', 'F1 fails only genes notebook 104 tested and found not to survive the core rebuild; an untested gene cannot fail.'),
    ('reading', 'F3 flags a gene whose mean edge enrichment over the relevant specimens is above the 95th percentile of its expression decile.'),
    ('reading', 'The C1 relabeled lines use the cutoff implied by each screen\'s own calls: the smallest |statistic| among called genes.'),
    ('reading', 'Line groups in Q2 are average-linkage clusters of 1 - phi cut at 0.5.'),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)

RUN = {'notebook': '112_pt_gene_atlas', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'guards': guard_log,
       'draws': DRAWS, 'aki_genes': int(len(AKI)), 'species_genes': int(len(SPC)),
       'aki_lines': {k: int(v) for k, v in aki_calls.sum().items()},
       'species_lines': {k: int(v) for k, v in spc_calls.sum().items()},
       'q1_holds': Q1_HOLDS, 'core': {k: int(len(v)) for k, v in CORE.items()},
       'novel': {k: int((~v['anchored']).sum()) for k, v in CORE.items()},
       'yield': YIELD.round(3).to_dict('records'),
       'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2, default=str))
print(json.dumps({k: RUN[k] for k in ('q1_holds', 'core', 'novel', 'seconds')}, indent=2, default=str))